Потренируемся самостоятельно писать многослойный перцептрон для работы с текстами.

Возьмем для этого датасет про юридические тексты. В этом датасете есть описания дел, а в качестве цп - то, что с делами произошло.

In [1]:
!wget https://raw.githubusercontent.com/rsuh-python/mag2022/main/CL/term02/06-Embeddings/legal_text_classification.csv

--2025-03-27 01:58:07--  https://raw.githubusercontent.com/rsuh-python/mag2022/main/CL/term02/06-Embeddings/legal_text_classification.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 68202412 (65M) [text/plain]
Saving to: ‘legal_text_classification.csv’

legal_text_classifi 100%[===================>]  65.04M   132MB/s    in 0.5s    

2025-03-27 01:58:08 (132 MB/s) - ‘legal_text_classification.csv’ saved [68202412/68202412]



Для начала напишем бейзлайн - логистическую регрессию. Возьмем в качестве признаков только текст - описание самого дела (case_text). Целевую переменную, очевидно, нужно превратить в чиселки (OHE).

- проверьте данные на пропуски
- проверьте баланс классов - это очень важно!
- используйте TF-IDF
- не забудьте использовать LabelEncoder
- логистической регрессии может понадобиться выставить solver='liblinear'
- если не помните, как работать с несбалансированными датасетами, просмотрите наши конспекты - точно где-то было (на худой конец документация к логрегу)

In [2]:
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder

In [3]:
data = pd.read_csv('legal_text_classification.csv')
data.head()

,case_id,case_outcome,case_title,case_text
0,Case1,cited,Alpine Hardwood (Aust) Pty Ltd v Hardys Pty Lt...,Ordinarily that discretion will be exercised s...
1,Case2,cited,Black v Lipovac [1998] FCA 699 ; (1998) 217 AL...,The general principles governing the exercise ...
2,Case3,cited,Colgate Palmolive Co v Cussons Pty Ltd (1993) ...,Ordinarily that discretion will be exercised s...
3,Case4,cited,Dais Studio Pty Ltd v Bullett Creative Pty Ltd...,The general principles governing the exercise ...
4,Case5,cited,Dr Martens Australia Pty Ltd v Figgins Holding...,The preceding general principles inform the ex...


In [ ]:
data.isna ()

,case_id,case_outcome,case_title,case_text
0,False,False,False,False
1,False,False,False,False
2,False,False,False,False
3,False,False,False,False
4,False,False,False,False
...,...,...,...,...
24980,False,False,False,False
24981,False,False,False,False
24982,False,False,False,False
24983,False,False,False,False


In [4]:
data.dropna ()
data.isna ()

,case_id,case_outcome,case_title,case_text
0,False,False,False,False
1,False,False,False,False
2,False,False,False,False
3,False,False,False,False
4,False,False,False,False
...,...,...,...,...
24980,False,False,False,False
24981,False,False,False,False
24982,False,False,False,False
24983,False,False,False,False


In [5]:
data.case_outcome.unique()

array(['cited', 'applied', 'followed', 'referred to', 'related',
       'considered', 'discussed', 'distinguished', 'affirmed', 'approved'],
      dtype=object)

In [6]:
data.case_outcome.value_counts()

,count
case_outcome,
cited,12219
referred to,4384
applied,2448
followed,2256
considered,1712
discussed,1024
distinguished,608
related,113
affirmed,113


In [7]:
data.drop(['case_title', 'case_id'], axis=1, inplace=True)
label_encoder = LabelEncoder()
data.case_outcome = label_encoder.fit_transform(data.case_outcome)



In [8]:
data.head ()

,case_outcome,case_text
0,3,Ordinarily that discretion will be exercised s...
1,3,The general principles governing the exercise ...
2,3,Ordinarily that discretion will be exercised s...
3,3,The general principles governing the exercise ...
4,3,The preceding general principles inform the ex...


In [9]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False)
encoded_features = encoder.fit_transform(data[['case_outcome']])
encoded_df = pd.DataFrame(encoded_features).astype (int)
df_final = pd.concat([data.drop(columns=['case_outcome']), encoded_df], axis=1)

df_final.head ()

,case_text,0,1,2,3,4,5,6,7,8,9
0,Ordinarily that discretion will be exercised s...,0,0,0,1,0,0,0,0,0,0
1,The general principles governing the exercise ...,0,0,0,1,0,0,0,0,0,0
2,Ordinarily that discretion will be exercised s...,0,0,0,1,0,0,0,0,0,0
3,The general principles governing the exercise ...,0,0,0,1,0,0,0,0,0,0
4,The preceding general principles inform the ex...,0,0,0,1,0,0,0,0,0,0


In [10]:
df_final1 = pd.concat ([data['case_outcome'], df_final], axis=1)
df_final1.head ()

,case_outcome,case_text,0,1,2,3,4,5,6,7,8,9
0,3,Ordinarily that discretion will be exercised s...,0,0,0,1,0,0,0,0,0,0
1,3,The general principles governing the exercise ...,0,0,0,1,0,0,0,0,0,0
2,3,Ordinarily that discretion will be exercised s...,0,0,0,1,0,0,0,0,0,0
3,3,The general principles governing the exercise ...,0,0,0,1,0,0,0,0,0,0
4,3,The preceding general principles inform the ex...,0,0,0,1,0,0,0,0,0,0


In [11]:
data.head ()

,case_outcome,case_text
0,3,Ordinarily that discretion will be exercised s...
1,3,The general principles governing the exercise ...
2,3,Ordinarily that discretion will be exercised s...
3,3,The general principles governing the exercise ...
4,3,The preceding general principles inform the ex...


In [12]:
X = data['case_text']
y = data['case_outcome']

In [13]:
data.dropna ()
data['case_text'] = data.astype(str).agg(''.join, axis=1)


In [14]:
print(data.isna().sum())

case_outcome    0
case_text       0
dtype: int64


In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

vect = TfidfVectorizer()
X_train1 = vect.fit_transform(X_train.values.astype('U'))
X_test1 = vect.transform(X_test.values.astype('U'))


model = LogisticRegression(solver='liblinear', class_weight='balanced')
model.fit(X_train1, y_train)


LogisticRegression(class_weight='balanced', solver='liblinear')

In [16]:
y_pred = model.predict (X_test1)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.24      0.69      0.35        32
           1       0.32      0.25      0.28       515
           2       0.11      0.32      0.17        19
           3       0.68      0.67      0.68      2457
           4       0.27      0.29      0.28       324
           5       0.19      0.34      0.24       205
           6       0.26      0.55      0.35       122
           7       0.34      0.27      0.30       436
           8       0.55      0.40      0.46       859
           9       0.26      0.54      0.35        28

    accuracy                           0.50      4997
   macro avg       0.32      0.43      0.35      4997
weighted avg       0.53      0.50      0.51      4997




Если все сделали как я, должна получиться средняя f-score в районе 0.5.

Теперь давайте попробуем написать нейронную сетку по аналогии с тетрадкой про твиттер из прошлого семинара.

In [17]:
import numpy as np
from string import punctuation
from collections import Counter
from sklearn.utils import shuffle, class_weight

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, RandomSampler, SequentialSampler
from torch.nn.utils.rnn import pad_sequence
import torch.optim as optim

class_weight - очень полезная для нас штука. Можно вычислить веса классов автоматически с ее помощью:

In [18]:
# первый аргумент - какие веса высчитывать, второй - какие у нас классы, третий - какие их частоты
yweights = class_weight.compute_class_weight('balanced', classes=np.unique(data.case_outcome), y=data.case_outcome)

Заметьте, что возвращает оно np.array.

Нужно написать:

- функцию для предобработки текста, которая получает сырой текст и возвращает список токенов
- создать словарь word2id
- и обратный ему id2word

In [19]:
def preprocess(text):
    tokens = text.lower().split()
    tokens = [token.strip(punctuation) for token in tokens]
    return tokens

from collections import Counter

vocab = Counter()

for text in data['case_text']:
    vocab.update(preprocess(text))
print('всего уникальных токенов:', len(vocab))

word2id = {'PAD': 0}

for word in vocab:
    word2id[word] = len(word2id)

id2word = {i: word for word, i in word2id.items()}

всего уникальных токенов: 67014


Лучше это все, конечно, запускать в колабе... не забудьте там выбрать T4 GPU в рантайме

In [20]:
DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
DEVICE

device(type='cuda')

Нужно написать класс для нашего датасета (можно беспощадно копипастить из тетрадки про твиттер)

In [21]:
class LegalDataset(Dataset):

    def __init__(self, dataset, word2id, DEVICE):
        self.dataset = dataset['text'].values
        self.word2id = word2id
        self.length = dataset.shape[0]
        self.target = dataset['tone'].values
        self.device = DEVICE

    def __len__(self): #это обязательный метод, он должен уметь считать длину датасета
        return self.length

    def __getitem__(self, index): #еще один обязательный метод. По индексу возвращает элемент выборки
        tokens = self.preprocess(self.dataset[index]) # токенизируем
        ids = torch.LongTensor([self.word2id[token] for token in tokens if token in self.word2id])
        y = [self.target[index]]
        return ids, y

    def preprocess(self, text):
        tokens = text.lower().split()
        tokens = [token.strip(punctuation) for token in tokens]
        tokens = [token for token in tokens if token]
        return tokens

    def collate_fn(self, batch): #этот метод можно реализовывать и отдельно,
    # он понадобится для DataLoader во время итерации по батчам
      ids, y = list(zip(*batch))
      padded_ids = pad_sequence(ids, batch_first=True).to(self.device)
      #мы хотим применять BCELoss, он будет брать на вход predicted размера batch_size x 1
      #(так как для каждого семпла модель будет отдавать одно число), target размера batch_size x 1
      y = torch.Tensor(y).to(self.device) # tuple ([1], [0], [1])  -> Tensor [[1.], [0.], [1.]]
      return padded_ids, y



In [22]:
train_sentences, val_sentences = train_test_split(data['case_text'], test_size=0.2)

In [23]:
print (train_sentences)

14215    4In the context of operations in overseas terr...
9147     3In Re Australian Elizabethan Theatre Trust; L...
20446    8The respondent stressed that the course taken...
24122    8As I have noted earlier, Div 7 of Pt VIB prov...
2134     3I accept the submission made by counsel for t...
                               ...                        
10996    8Support for the view that a distinction is to...
672      7Tribunal saw the issue between the parties as...
11304    3The reasonable anticipation test is not a que...
22457    3No doubt, many descriptions of the purpose of...
12223    8a cause of action may be viable in the United...
Name: case_text, Length: 19988, dtype: object


In [29]:
word2id.update({'<UNK>': -1})

In [32]:
def text_to_ids(text, word2id, max_len=6):
    tokens = text.lower().split()
    ids = [word2id.get(t, word2id["<UNK>"]) for t in tokens]
    ids = ids[:max_len] + [0] * (max_len - len(ids))
    return ids

max_len = 90

data_copy = data

data_copy["ids"] = data_copy["case_text"].apply(lambda t: text_to_ids(t, word2id, max_len=max_len))

id_cols = data_copy["ids"].apply(pd.Series)
id_cols.columns = [f"tok_{i+1}" for i in id_cols.columns]

big_df = pd.concat([data_copy, id_cols], axis=1).drop(columns=["ids"])
big_df.head ()

,case_outcome,case_text,tok_1,tok_2,tok_3,tok_4,tok_5,tok_6,tok_7,tok_8,...,tok_81,tok_82,tok_83,tok_84,tok_85,tok_86,tok_87,tok_88,tok_89,tok_90
0,3,3Ordinarily that discretion will be exercised ...,1,2,3,4,5,6,7,2,...,0,0,0,0,0,0,0,0,0,0
1,3,3The general principles governing the exercise...,65,66,67,68,10,69,70,10,...,110,37,38,39,111,112,37,38,-1,44
2,3,3Ordinarily that discretion will be exercised ...,1,2,3,4,5,6,7,2,...,0,0,0,0,0,0,0,0,0,0
3,3,3The general principles governing the exercise...,65,66,67,68,10,69,70,10,...,110,37,38,39,111,112,37,38,-1,44
4,3,3The preceding general principles inform the e...,65,147,66,67,148,10,69,70,...,2,16,181,182,5,-1,10,177,184,10


In [36]:
big_df_copy = big_df
big_df_copy = big_df_copy.drop (columns=["case_text"])

In [37]:
train_sentences, val_sentences = train_test_split(big_df_copy.drop (columns=["case_outcome"]), test_size=0.2)

In [38]:
train_sentences.head ()

,tok_1,tok_2,tok_3,tok_4,tok_5,tok_6,tok_7,tok_8,tok_9,tok_10,...,tok_81,tok_82,tok_83,tok_84,tok_85,tok_86,tok_87,tok_88,tok_89,tok_90
3727,-1,10,84,1977,2,551,23920,656,12,1522,...,1272,61,-1,5752,53,232,16,427,102,10576
4123,1000,722,29,3581,182,102,265,10,7504,119,...,2,119,98,204,3581,-1,32,368,868,204
23922,6658,-1,2193,135,2192,15,10,1248,70,6468,...,-1,901,61,-1,1714,50,999,10,316,70
18814,1000,1093,106,5,3280,235,-1,-1,-1,325,...,23,74,452,-1,185,161,8998,-1,161,-1
23413,65001,70,2765,10062,32,452,268,135,1440,73,...,185,4536,50,1657,852,699,12,1723,-1,12


In [40]:
train_sampler = RandomSampler(train_sentences)
train_iterator = DataLoader(train_sentences, sampler=train_sampler, batch_size=1024)

In [43]:
val_sampler = RandomSampler(val_sentences)
val_iterator = DataLoader(val_sentences, sampler=val_sampler, batch_size=1024)

In [84]:
X_train = val_sentences
X_test = val_sentences
y_train, y_test = train_test_split(y, test_size=0.2)

In [57]:
from sklearn.preprocessing import StandardScaler

X = big_df_copy.drop (columns=["case_outcome"])
y = big_df_copy["case_outcome"]

y.head ()

,case_outcome
0,3
1,3
2,3
3,3
4,3


In [82]:
y = y.astype(int)

In [58]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [59]:
X_train, X_test = train_test_split(X_scaled, test_size=0.2)

In [99]:
from torch.utils.data import TensorDataset, DataLoader

X_tensor = torch.tensor(X_train, dtype=torch.float32)
y_tensor = torch.tensor(y_train, dtype=torch.float32)

dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(dataset, batch_size=512, shuffle=True)

ValueError: could not determine the shape of object type 'DataFrame'

Ну и наконец напишем архитектуру. Модель при инициализации должна принимать размер словаря и эмбеддинга. У нас в датасете 10 классов, поэтому, в отличие от тетрадки про твиттер, нужно использовать Softmax и возвращать вероятности классов. В качестве лосса подойдет кросс-энтропия (я ее уже за вас вписала вместе с весами классов).

In [52]:
import torch
import torch.nn as nn
import torch.nn.functional as F

num_classes = 10
input_dim = 90
learning_rate = 0.01
epochs = 100


class MLP (nn.Module):
    def __init__(self, input_dim, num_classes, hidden_dim=128):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        logits = self.fc2(x)
        probs = F.softmax(logits, dim=1)
        return probs

In [54]:
model = MLP(input_dim, num_classes)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

Теперь нужно написать трейнлуп (лучше скопипастить откуда-нибудь), инициализировать нашу модель и запустить)

In [97]:
def train(model, train_loader, val_loader, optimizer, device, class_weights, epochs):
    model.to(device)


    if class_weights is not None:
        class_weights = class_weights.to(device)
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
    else:
        loss_fn = torch.nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        correct = 0
        total = 0

        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            y = y.long ()

            optimizer.zero_grad()
            logits = model(X)
            loss = loss_fn(logits, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * X.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

        avg_loss = total_loss / total
        acc = correct / total * 100
        print(f"[Epoch {epoch+1}] Loss: {avg_loss:.4f} | Acc: {acc:.2f}%")

        if val_loader is not None:
            model.eval()
            val_loss = 0
            val_correct = 0
            val_total = 0
            with torch.no_grad():
                for X_val, y_val in val_loader:
                    y_val = y_val.long ()
                    X_val, y_val = X_val.to(device), y_val.to(device)
                    logits = model(X_val)
                    loss = loss_fn(logits, y_val)
                    val_loss += loss.item() * X_val.size(0)
                    val_correct += (logits.argmax(dim=1) == y_val).sum().item()
                    val_total += y_val.size(0)

            val_loss /= val_total
            val_acc = val_correct / val_total * 100
            print(f"           Val Loss: {val_loss:.4f} | Acc: {val_acc:.2f}%")

In [64]:
X_test

,tok_1,tok_2,tok_3,tok_4,tok_5,tok_6,tok_7,tok_8,tok_9,tok_10,...,tok_81,tok_82,tok_83,tok_84,tok_85,tok_86,tok_87,tok_88,tok_89,tok_90
10282,13952,10,450,70,10,1032,135,98,1999,70,...,893,98,2838,2,16,883,23,1474,16,228
11530,6019,13,50,529,2900,1891,15,10,635,180,...,70,10,139,102,265,10,471,119,-1,2250
23091,1799,180,23,16914,119,1895,73,10,770,5373,...,105,119,351,119,98,-1,117,243,97,98
23904,11255,67,13,2444,32,103,33,119,609,20,...,0,0,0,0,0,0,0,0,0,0
16315,1718,-1,16,83,84,249,598,32,16346,39,...,2153,20,1632,235,23,948,92,1372,1603,129
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16182,65,207,2102,-1,3906,-1,280,13,4409,1633,...,6493,39,52685,-1,1059,48,9237,50,8918,-1
22370,63586,224,70,103,84,2102,2,1093,2,13,...,1004,630,271,1069,10,5783,84,97,98,291
16409,11662,10,569,10,67,325,265,289,135,3074,...,13,246,273,2036,70,252,12,268,828,52938
13164,1555,1158,5342,3940,57,38,-1,939,845,-1,...,119,3633,2,615,678,608,32,5618,-1,235


In [65]:
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)

dataset2 = TensorDataset(X_test_tensor, y_test_tensor)
val_loader = DataLoader(dataset, batch_size=512, shuffle=True)

In [80]:
yweights_tensor = torch.from_numpy(yweights).float().to(device)

In [78]:
print (yweights)

[22.11061947  1.02062908 23.13425926  0.20447663  1.45940421  2.43994141
  4.109375    1.10749113  0.56991332 22.11061947]


In [98]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train (model, train_loader, val_loader, optimizer, device, yweights_tensor, epochs)

[Epoch 1] Loss: 2.2317 | Acc: 9.48%
           Val Loss: 2.1843 | Acc: 9.09%
[Epoch 2] Loss: 2.2028 | Acc: 9.70%
           Val Loss: 2.1664 | Acc: 9.86%
[Epoch 3] Loss: 2.1656 | Acc: 11.29%
           Val Loss: 2.1372 | Acc: 13.68%
[Epoch 4] Loss: 2.1438 | Acc: 12.96%
           Val Loss: 2.1171 | Acc: 13.89%
[Epoch 5] Loss: 2.1229 | Acc: 14.09%
           Val Loss: 2.0974 | Acc: 16.02%
[Epoch 6] Loss: 2.1111 | Acc: 15.31%
           Val Loss: 2.0892 | Acc: 14.66%
[Epoch 7] Loss: 2.1136 | Acc: 15.03%
           Val Loss: 2.0882 | Acc: 14.88%
[Epoch 8] Loss: 2.0920 | Acc: 16.17%
           Val Loss: 2.0767 | Acc: 14.63%
[Epoch 9] Loss: 2.0882 | Acc: 17.45%
           Val Loss: 2.0662 | Acc: 18.61%
[Epoch 10] Loss: 2.0797 | Acc: 16.71%
           Val Loss: 2.0561 | Acc: 20.40%
[Epoch 11] Loss: 2.0722 | Acc: 18.30%
           Val Loss: 2.0600 | Acc: 21.31%
[Epoch 12] Loss: 2.0617 | Acc: 19.59%
           Val Loss: 2.0481 | Acc: 17.62%
[Epoch 13] Loss: 2.0682 | Acc: 17.21%
           Val 

Скорее всего, вам понадобится учиться очень много эпох, чтобы предсказывать что-нибудь стоящее (эпох 100...), и, вероятнее всего, придется играться с архитектурой, чтобы получить приличное качество. На семинаре на эксперименты времени нет, поэтому добаловаться можно дома - и заодно попробовать подключить эмбеддинги w2v, например.

In [101]:
train_sampler = RandomSampler(train_sentences)
train_iterator = DataLoader(train_sentences, sampler=train_sampler, batch_size=256)

val_sampler = RandomSampler(val_sentences)
val_iterator = DataLoader(val_sentences, sampler=val_sampler, batch_size=256)

X_train, X_test = train_test_split(X_scaled, test_size=0.2)

X_tensor = torch.tensor(X_train, dtype=torch.float32)
y_tensor = torch.tensor(y_train, dtype=torch.float32)

dataset = TensorDataset(X_tensor, y_tensor)
train_loader = DataLoader(dataset, batch_size=256, shuffle=True)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32)

dataset2 = TensorDataset(X_test_tensor, y_test_tensor)
val_loader = DataLoader(dataset, batch_size=256, shuffle=True)

In [102]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train (model, train_loader, val_loader, optimizer, device, yweights_tensor, epochs)

[Epoch 1] Loss: 2.3552 | Acc: 13.79%
           Val Loss: 2.3421 | Acc: 15.78%
[Epoch 2] Loss: 2.3454 | Acc: 19.52%
           Val Loss: 2.3292 | Acc: 22.20%
[Epoch 3] Loss: 2.3338 | Acc: 22.98%
           Val Loss: 2.3237 | Acc: 25.94%
[Epoch 4] Loss: 2.3273 | Acc: 21.99%
           Val Loss: 2.3177 | Acc: 24.32%
[Epoch 5] Loss: 2.3213 | Acc: 20.66%
           Val Loss: 2.3135 | Acc: 20.55%
[Epoch 6] Loss: 2.3175 | Acc: 23.23%
           Val Loss: 2.3098 | Acc: 26.07%
[Epoch 7] Loss: 2.3140 | Acc: 24.17%
           Val Loss: 2.3101 | Acc: 23.66%
[Epoch 8] Loss: 2.3110 | Acc: 22.77%
           Val Loss: 2.3032 | Acc: 23.19%
[Epoch 9] Loss: 2.3049 | Acc: 23.88%
           Val Loss: 2.3014 | Acc: 21.08%
[Epoch 10] Loss: 2.3024 | Acc: 22.45%
           Val Loss: 2.2976 | Acc: 21.75%
[Epoch 11] Loss: 2.3007 | Acc: 18.97%
           Val Loss: 2.2943 | Acc: 20.17%
[Epoch 12] Loss: 2.3007 | Acc: 21.50%
           Val Loss: 2.2933 | Acc: 21.60%
[Epoch 13] Loss: 2.2988 | Acc: 20.56%
           

Наверное, стоит увеличить датасет, который мы загружаем, потому что я обрезал его на 90 словах для скорости. Тем временем, мы видели, что там бывают вхождения на 20000 слов...

In [109]:
from sklearn.metrics import f1_score

def train(model, train_loader, val_loader, optimizer, device, class_weights, epochs):
    model.to(device)


    if class_weights is not None:
        class_weights = class_weights.to(device)
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
    else:
        loss_fn = torch.nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        correct = 0
        total = 0

        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            y = y.long ()

            optimizer.zero_grad()
            logits = model(X)
            loss = loss_fn(logits, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * X.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)

        avg_loss = total_loss / total
        acc = correct / total * 100
        print(f"[Epoch {epoch+1}] Loss: {avg_loss:.4f} | Acc: {acc:.2f}%")

        if val_loader is not None:
            model.eval()
            val_loss = 0
            val_correct = 0
            val_total = 0

            y_true_all = []
            y_pred_all = []

            with torch.no_grad():
                for X_val, y_val in val_loader:
                    y_val = y_val.long()
                    X_val, y_val = X_val.to(device), y_val.to(device)
                    logits = model(X_val)
                    loss = loss_fn(logits, y_val)
                    val_loss += loss.item() * X_val.size(0)

                    preds = logits.argmax(dim=1)
                    val_correct += (preds == y_val).sum().item()
                    val_total += y_val.size(0)

                    y_true_all.extend(y_val.cpu().numpy())
                    y_pred_all.extend(preds.cpu().numpy())

            val_loss /= val_total
            val_acc = val_correct / val_total * 100
            f1 = f1_score(y_true_all, y_pred_all, average='weighted')

            print(f"           Val Loss: {val_loss:.4f} | Acc: {val_acc:.2f}% | F1: {f1:.4f}")

In [110]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train (model, train_loader, val_loader, optimizer, device, yweights_tensor, epochs)

[Epoch 1] Loss: 2.2070 | Acc: 37.16%
           Val Loss: 2.2066 | Acc: 37.80% | F1: 0.3879
[Epoch 2] Loss: 2.2102 | Acc: 35.37%
           Val Loss: 2.2074 | Acc: 37.36% | F1: 0.3831
[Epoch 3] Loss: 2.2069 | Acc: 37.06%
           Val Loss: 2.2043 | Acc: 38.52% | F1: 0.3947
[Epoch 4] Loss: 2.2064 | Acc: 37.46%
           Val Loss: 2.2028 | Acc: 38.66% | F1: 0.3958
[Epoch 5] Loss: 2.2063 | Acc: 37.98%
           Val Loss: 2.2032 | Acc: 37.37% | F1: 0.3828
[Epoch 6] Loss: 2.2052 | Acc: 38.09%
           Val Loss: 2.2030 | Acc: 38.39% | F1: 0.3927
[Epoch 7] Loss: 2.2020 | Acc: 38.33%
           Val Loss: 2.2000 | Acc: 38.88% | F1: 0.3981
[Epoch 8] Loss: 2.2042 | Acc: 38.32%
           Val Loss: 2.2034 | Acc: 39.16% | F1: 0.3989
[Epoch 9] Loss: 2.2081 | Acc: 37.88%
           Val Loss: 2.2039 | Acc: 36.98% | F1: 0.3786
[Epoch 10] Loss: 2.2031 | Acc: 37.74%
           Val Loss: 2.2070 | Acc: 35.44% | F1: 0.3693
[Epoch 11] Loss: 2.2089 | Acc: 36.44%
           Val Loss: 2.2058 | Acc: 36.56%